# Statistical Fishing Effort Classification

This notebook demonstrates the **rule-based statistical classifier** for fishing effort.
Unlike the ML model, this approach:
- ✅ **No training required** - uses empirically-derived behavioral thresholds
- ✅ **Explainable** - clear rules based on fishing vessel behavior
- ✅ **Lightweight** - only requires timestamp, lat, lon (and optional trip_id)
- ✅ **Customizable** - thresholds can be tuned for different vessel types

## Behavioral Indicators Used:
1. **Speed patterns**: Fishing occurs at 0.5-8 km/h (gear deployment/retrieval)
2. **Turning behavior**: High turning angles indicate searching/maneuvering
3. **Path straightness**: Low straightness = non-linear fishing operations
4. **Spatial clustering**: Repeated operations in same area
5. **Speed variability**: Start-stop behavior during fishing
6. **Sinuosity**: Tortuous paths indicate active fishing

## 1. Setup & Imports

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from ssfaitk.utils.plot_trip_route import plot_trip_route, plot_trip_route_png

from ssfaitk.utils.trip_file_loader import load_random_trips

# Statistical effort classifier
from ssfaitk.models.effort.statistical_effort import StatisticalEffortClassifier as old_SEC
from ssfaitk.models.effort.statistical_effort_v2 import StatisticalEffortClassifier as new_SEC

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

saving_output_dir = "test_outputs"
# Create output directory
output_dir = Path(saving_output_dir)
output_dir.mkdir(exist_ok=True)

## 2. Load Your GPS Data

**Minimum required columns:**
- `timestamp`: Date/time of GPS point (any parseable format)
- `latitude`: Latitude in decimal degrees
- `longitude`: Longitude in decimal degrees
- `trip_id` (optional): Trip identifier for multi-trip datasets

In [2]:
# Load your data
df = load_random_trips(10, seed=42)
print(f"✓ Loaded {df['Trip'].nunique()} trips!")

print(f"✓ Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
print("\nFirst few rows:")
df.head()

Found 8223 trips in /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar
Randomly selected 10 trips (seed=42)
Selected trip IDs: ['14517537' '14382277' '13570346' '13260243' '14460963' '13919942'
 '14096904' '13221816' '14295907' '12965724']
Loading 10 trips...
Loading trip 14517537 from /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar/pds-tracks_14517537.parquet
Loaded 5,150 points for trip 14517537
  [1/10] Trip 14517537: 5,150 points ✓
Loading trip 14382277 from /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar/pds-tracks_14382277.parquet
Loaded 433 points for trip 14382277
  [2/10] Trip 14382277: 433 points ✓
Loading trip 13570346 from /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar/pds-tracks_13570346.parquet
Loaded 694 points for trip 13570346
  [3/10] Trip 13570346: 694 points ✓
Loading trip 13260243 from /Users/altarturhamza/Documents/G

✓ Loaded 10 trips!
✓ Data loaded: 25,975 rows, 10 columns

Columns: ['Time', 'Boat', 'Trip', 'Lat', 'Lng', 'Speed (M/S)', 'Range (Meters)', 'Heading', 'Boat Name', 'Community']

First few rows:


,Time,Boat,Trip,Lat,Lng,Speed (M/S),Range (Meters),Heading,Boat Name,Community
0,2026-01-26 23:23:00+00:00,24539.0,1.4518e+07,-4.9009,39.7327,0.00,0.0000,49.0,Kwa roho safi,"Msuka Magharibi,2026-01-29 09:29:10+00,2026-01..."
1,2026-01-26 23:28:45+00:00,24539.0,1.4518e+07,-4.9009,39.7327,0.01,5.9575,74.0,Kwa roho safi,"Msuka Magharibi,2026-01-29 09:29:10+00,2026-01..."
2,2026-01-26 23:34:30+00:00,24539.0,1.4518e+07,-4.9009,39.7327,0.01,4.9573,44.0,Kwa roho safi,"Msuka Magharibi,2026-01-29 09:29:10+00,2026-01..."
3,2026-01-26 23:40:15+00:00,24539.0,1.4518e+07,-4.9009,39.7327,0.00,7.0937,54.0,Kwa roho safi,"Msuka Magharibi,2026-01-29 09:29:10+00,2026-01..."
4,2026-01-26 23:46:00+00:00,24539.0,1.4518e+07,-4.9009,39.7328,0.02,2.4787,46.0,Kwa roho safi,"Msuka Magharibi,2026-01-29 09:29:10+00,2026-01..."


## 3. Quick Start - One-Line Prediction

The simplest way to get predictions:

In [3]:
old_clf = old_SEC()

# One-line prediction with default settings
predictions_simple = old_clf.predict(df)

print("\n✓ Predictions complete!")
print("\nFishing activity distribution:")
print(predictions_simple['is_fishing'].value_counts(normalize=True))

Resolved columns: lat=Lat, lon=Lng, time=Time, trip=Trip
Processing 25975 points across 10 trips
Computing kinematic features...
Computing local statistics...
Computing spatial features...
Computing temporal features...
Applying statistical classification rules...
Classification complete: 19.3% classified as fishing



✓ Predictions complete!

Fishing activity distribution:
is_fishing
0    0.8065
1    0.1935
Name: proportion, dtype: float64


## 4. Detailed Workflow - Full Control

For more control over the process:

In [4]:
# Initialize classifier

print("Default classification thresholds:")
print(f"  Fishing speed range: {old_clf.config['min_fishing_speed']}-{old_clf.config['max_fishing_speed']} km/h")
print(f"  High turn threshold: {old_clf.config['high_turn_threshold']}°")
print(f"  Low straightness threshold: {old_clf.config['low_straightness_threshold']}")
print(f"  Fishing score threshold: {old_clf.config['fishing_score_threshold']}")

Default classification thresholds:
  Fishing speed range: 0.5-8.0 km/h
  High turn threshold: 45.0°
  Low straightness threshold: 0.4
  Fishing score threshold: 0.5


In [5]:
trips = list(set(list(predictions_simple['trip_id'])))
base_dir = 'effort prediction visuals'
for TRP in trips:
    plot_trip_route(predictions_simple, trip_id=TRP, output_path=f'{base_dir}/htmls/old/trip {TRP}.html')
    plot_trip_route_png(predictions_simple, trip_id=TRP, output_path=f'{base_dir}/images/old/trip {TRP} - with stats.png')


Plotting trip 14517537.0 with 5150 points...
✓ Map saved to: effort prediction visuals/htmls/old/trip 14517537.0.html
  Total points: 5150
  Fishing: 1702 (33.0%)
  Duration: 57.3 hours
  Distance: 125.2 km
Plotting trip 14517537.0 with 5150 points...
✓ PNG map saved to: effort prediction visuals/images/old/trip 14517537.0 - with stats.png
  Total points: 5150
  Fishing: 1702 (33.0%)
  Duration: 57.3 hours
  Distance: 125.2 km
Plotting trip 14295907.0 with 2132 points...
✓ Map saved to: effort prediction visuals/htmls/old/trip 14295907.0.html
  Total points: 2132
  Fishing: 399 (18.7%)
  Duration: 14.3 hours
  Distance: 37.2 km
Plotting trip 14295907.0 with 2132 points...
✓ PNG map saved to: effort prediction visuals/images/old/trip 14295907.0 - with stats.png
  Total points: 2132
  Fishing: 399 (18.7%)
  Duration: 14.3 hours
  Distance: 37.2 km
Plotting trip 14460963.0 with 3082 points...
✓ Map saved to: effort prediction visuals/htmls/old/trip 14460963.0.html
  Total points: 3082
  F

### 4.1 Run Prediction

In [6]:
new_clf = new_SEC()

# One-line prediction with default settings
predictions_simple_v2 = new_clf.predict(df)

print("\n✓ Predictions complete!")
print("\nFishing activity distribution:")
print(predictions_simple_v2['is_fishing'].value_counts(normalize=True))

Resolved columns: lat=Lat, lon=Lng, time=Time, trip=Trip
Processing 25975 points across 10 trips
Computing kinematic features (with improved turn angle filtering)...
Computing local statistics (time-based windows: [10.0] min)...
Computing spatial features (distance-based window: 1.0 km)...
Computing temporal features...
Applying statistical classification rules (with feature interactions)...
Classification complete: 9.6% classified as fishing
State machine smoothing applied (min duration: 3 points)



✓ Predictions complete!

Fishing activity distribution:
is_fishing
0    0.9035
1    0.0965
Name: proportion, dtype: float64


### 4.2 Examine Fishing Indicators

Look at the individual behavioral indicators that contribute to the classification:

In [8]:
# Initialize classifier

print("Default classification thresholds:")
print(f"  Fishing speed range: {new_clf.config['min_fishing_speed']}-{new_clf.config['max_fishing_speed']} km/h")
print(f"  High turn threshold: {new_clf.config['high_turn_threshold']}°")
print(f"  Low straightness threshold: {new_clf.config['low_straightness_threshold']}")
print(f"  Fishing score threshold: {new_clf.config['fishing_score_threshold']}")

Default classification thresholds:
  Fishing speed range: 0.5-8.0 km/h
  High turn threshold: 45.0°
  Low straightness threshold: 0.4
  Fishing score threshold: 0.5


In [14]:
trips = list(set(list(predictions_simple_v2['trip_id'])))
base_dir = 'effort prediction visuals'
for TRP in trips:
    plot_trip_route(predictions_simple_v2, trip_id=TRP, output_path=f'{base_dir}/htmls/new/trip {TRP}.html')
    plot_trip_route_png(predictions_simple_v2, trip_id=TRP, output_path=f'{base_dir}/images/new/trip {TRP} - with stats.png')

Plotting trip 14517537.0 with 5150 points...
✓ Map saved to: effort prediction visuals/htmls/new/trip 14517537.0.html
  Total points: 5150
  Fishing: 837 (16.3%)
  Duration: 57.3 hours
  Distance: 125.2 km
Plotting trip 14517537.0 with 5150 points...
✓ PNG map saved to: effort prediction visuals/images/new/trip 14517537.0 - with stats.png
  Total points: 5150
  Fishing: 837 (16.3%)
  Duration: 57.3 hours
  Distance: 125.2 km
Plotting trip 14295907.0 with 2132 points...
✓ Map saved to: effort prediction visuals/htmls/new/trip 14295907.0.html
  Total points: 2132
  Fishing: 295 (13.8%)
  Duration: 14.3 hours
  Distance: 37.2 km
Plotting trip 14295907.0 with 2132 points...
✓ PNG map saved to: effort prediction visuals/images/new/trip 14295907.0 - with stats.png
  Total points: 2132
  Fishing: 295 (13.8%)
  Duration: 14.3 hours
  Distance: 37.2 km
Plotting trip 14460963.0 with 3082 points...
✓ Map saved to: effort prediction visuals/htmls/new/trip 14460963.0.html
  Total points: 3082
  Fis

In [10]:
MODIRATE_CONFIG = {
    'fishing_score_threshold': 0.40,      # Lower = more detection
    # 'min_state_duration': 1,              # Keep short fishing events
    'min_fishing_speed': 0.2,             # Include slower activity
    # 'max_fishing_speed': 10.0,            # Include faster activity  
    'high_turn_threshold': 40.0,          # Gentler turns count
    'clustering_radius_km': 0.6,          # Wider clustering area
    'low_straightness_threshold': 0.45,    # Less meandering needed
}
clf_v2_editedConfig = new_SEC(config=MODIRATE_CONFIG)
predictions_simple_v2_editedConfig = clf_v2_editedConfig.predict(df)
print("\n✓ Predictions complete!")
print("\nFishing activity distribution:")
print(predictions_simple_v2_editedConfig['is_fishing'].value_counts(normalize=True))

Resolved columns: lat=Lat, lon=Lng, time=Time, trip=Trip
Processing 25975 points across 10 trips
Computing kinematic features (with improved turn angle filtering)...
Computing local statistics (time-based windows: [10.0] min)...
Computing spatial features (distance-based window: 1.0 km)...
Computing temporal features...
Applying statistical classification rules (with feature interactions)...
Classification complete: 21.6% classified as fishing
State machine smoothing applied (min duration: 3 points)



✓ Predictions complete!

Fishing activity distribution:
is_fishing
0    0.7844
1    0.2156
Name: proportion, dtype: float64


In [12]:
# Initialize classifier

print("Default classification thresholds:")
print(f"  Fishing speed range: {clf_v2_editedConfig.config['min_fishing_speed']}-{clf_v2_editedConfig.config['max_fishing_speed']} km/h")
print(f"  High turn threshold: {clf_v2_editedConfig.config['high_turn_threshold']}°")
print(f"  Low straightness threshold: {clf_v2_editedConfig.config['low_straightness_threshold']}")
print(f"  Fishing score threshold: {clf_v2_editedConfig.config['fishing_score_threshold']}")

Default classification thresholds:
  Fishing speed range: 0.2-8.0 km/h
  High turn threshold: 40.0°
  Low straightness threshold: 0.45
  Fishing score threshold: 0.4


In [15]:
trips = list(set(list(predictions_simple_v2_editedConfig['trip_id'])))
base_dir = 'effort prediction visuals'
for TRP in trips:
    plot_trip_route(predictions_simple_v2_editedConfig, trip_id=TRP, output_path=f'{base_dir}/htmls/new/trip {TRP} v2.html')
    plot_trip_route_png(predictions_simple_v2_editedConfig, trip_id=TRP, output_path=f'{base_dir}/images/new/trip {TRP} - with stats v2.png')

Plotting trip 14517537.0 with 5150 points...
✓ Map saved to: effort prediction visuals/htmls/new/trip 14517537.0 v2.html
  Total points: 5150
  Fishing: 1979 (38.4%)
  Duration: 57.3 hours
  Distance: 125.2 km
Plotting trip 14517537.0 with 5150 points...
✓ PNG map saved to: effort prediction visuals/images/new/trip 14517537.0 - with stats v2.png
  Total points: 5150
  Fishing: 1979 (38.4%)
  Duration: 57.3 hours
  Distance: 125.2 km
Plotting trip 14295907.0 with 2132 points...
✓ Map saved to: effort prediction visuals/htmls/new/trip 14295907.0 v2.html
  Total points: 2132
  Fishing: 388 (18.2%)
  Duration: 14.3 hours
  Distance: 37.2 km
Plotting trip 14295907.0 with 2132 points...
✓ PNG map saved to: effort prediction visuals/images/new/trip 14295907.0 - with stats v2.png
  Total points: 2132
  Fishing: 388 (18.2%)
  Duration: 14.3 hours
  Distance: 37.2 km
Plotting trip 14460963.0 with 3082 points...
✓ Map saved to: effort prediction visuals/htmls/new/trip 14460963.0 v2.html
  Total p